# 第三十三课(毕业设计 II):纯 numpy 跑真模型

讲义:[33-numpy-llm.md](33-numpy-llm.md)。safetensors 逐张量加载 SmolLM2-135M + BPE 分词 + Llama 前向(GQA/每头 RoPE/SwiGLU/KV cache),真实生成英文。

In [ ]:
# ===== 加载器 + 分词器(讲义 §2–3) =====
import glob
import json
import struct
import time
import numpy as np
import regex as re

SNAP = glob.glob("/Volumes/Sam-APFS/huggingface/hub/models--HuggingFaceTB--"
                 "SmolLM2-135M-Instruct/snapshots/*")[0]
ST = f"{SNAP}/model.safetensors"

def read_header(path):
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        return json.loads(f.read(n))

HEADER = read_header(ST)
with open(ST, "rb") as f:
    _HDRL = struct.unpack("<Q", f.read(8))[0]

def load(name):
    meta = HEADER[name]
    beg, end = meta["data_offsets"]
    with open(ST, "rb") as f:
        f.seek(8 + _HDRL + beg)
        raw = f.read(end - beg)
    arr = np.frombuffer(raw, dtype=np.uint16 if meta["dtype"] == "BF16" else np.float32)
    if meta["dtype"] == "BF16":
        arr = (arr.astype(np.uint32) << 16).view(np.float32)   # bf16 = fp32 高 16 位
    return arr.reshape(meta["shape"])

class Tokenizer:
    """22 课的 BPE 思想 + 本模型 vocab/merges;GPT-2 预切分需要 regex 的 \p 类。"""
    SPLIT = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

    def __init__(self, tj):
        self.vocab = tj["model"]["vocab"]
        self.id_to_tok = {v: k for k, v in self.vocab.items()}
        self.ranks = {tuple(m.split(" ")): i for i, m in enumerate(tj["model"]["merges"])}
        bs = (list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)))
        cs = bs[:]
        n = 0
        for b in range(256):
            if b not in bs:
                bs.append(b); cs.append(256 + n); n += 1
        self.B2U = {b: chr(c) for b, c in zip(bs, cs)}
        self.U2B = {v: k for k, v in self.B2U.items()}
        for t in tj.get("added_tokens", []):
            self.id_to_tok[t["id"]] = t["content"]

    def _bpe(self, word):
        syms = list(word)
        while len(syms) > 1:
            best_k, best_rank = None, None
            for k, p in enumerate(zip(syms, syms[1:])):
                r = self.ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_k, best_rank = k, r
            if best_k is None:
                return syms
            syms[best_k:best_k + 2] = [syms[best_k] + syms[best_k + 1]]
        return syms

    def encode(self, text):
        ids = []
        for piece in self.SPLIT.findall(text):
            w = "".join(self.B2U[b] for b in piece.encode("utf-8"))
            for s in self._bpe(w):
                ids.append(self.vocab[s])
        return ids

    def decode(self, ids):
        s = "".join(self.id_to_tok.get(i, "") for i in ids)
        return bytes(self.U2B[c] for c in s).decode("utf-8", errors="replace")

In [ ]:
# ===== Llama 前向(讲义 §4;每头内 RoPE——见 §5 战争故事) =====
def softmax(x, axis=-1):
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)

class SmolLM2:
    def __init__(self):
        self.cfg = json.load(open(f"{SNAP}/config.json", encoding="utf-8"))
        self.H = self.cfg["hidden_size"]
        self.NL = self.cfg["num_hidden_layers"]
        self.NQ = self.cfg["num_attention_heads"]
        self.NKV = self.cfg["num_key_value_heads"]
        self.DH = self.H // self.NQ
        self.theta = self.cfg["rope_theta"]
        self.eps = self.cfg["rms_norm_eps"]
        t0 = time.time()
        self.W = {name: load(name) for name in HEADER if name != "__metadata__"}
        self.load_s = time.time() - t0

    def rms(self, x, g):
        return x / np.sqrt((x ** 2).mean(-1, keepdims=True) + self.eps) * g

    def rope(self, x, pos):
        """x: [..., dh](逐头);半分配对在每个头内部——不能在展平 [T,H] 上跨头配对。"""
        half = x.shape[-1] // 2
        inv = 1.0 / (self.theta ** (np.arange(half) * 2 / x.shape[-1]))
        ang = pos.reshape(-1, *([1] * (x.ndim - 1))) * inv
        c, s = np.cos(ang), np.sin(ang)
        x1, x2 = x[..., :half], x[..., half:]
        return np.concatenate([x1 * c - x2 * s, x1 * s + x2 * c], -1)

    def forward(self, ids, cache=None):
        pos0 = cache[0][0].shape[1] if cache else 0
        T = len(ids)
        W = self.W
        x = W["model.embed_tokens.weight"][np.array(ids)]
        new_cache = []
        mask = np.triu(np.full((T, T + pos0), -np.inf), 1 + pos0)
        for i in range(self.NL):
            p = f"model.layers.{i}."
            h = self.rms(x, W[p + "input_layernorm.weight"])
            nq, nkv, dh = self.NQ, self.NKV, self.DH
            pos = np.arange(pos0, pos0 + T)
            q = self.rope((h @ W[p + "self_attn.q_proj.weight"].T).reshape(T, nq, dh), pos)
            k = self.rope((h @ W[p + "self_attn.k_proj.weight"].T).reshape(T, nkv, dh), pos)
            v = h @ W[p + "self_attn.v_proj.weight"].T
            k = k.transpose(1, 0, 2)                    # [NKV, T, DH]
            v = v.reshape(T, nkv, dh).transpose(1, 0, 2)
            if cache is not None:
                k = np.concatenate([cache[i][0], k], 1)
                v = np.concatenate([cache[i][1], v], 1)
            new_cache.append((k, v))
            q = q.transpose(1, 0, 2)                    # [NQ, T, DH]
            reps = nq // nkv                            # GQA:每 KV 头服务 3 个 Q 头
            outs = []
            for hq in range(nq):
                kvh = hq // reps
                s = q[hq] @ k[kvh].T / np.sqrt(dh) + mask
                outs.append(softmax(s) @ v[kvh])
            attn = np.stack(outs, 0).transpose(1, 0, 2).reshape(T, self.H)
            x = x + attn @ W[p + "self_attn.o_proj.weight"].T
            h = self.rms(x, W[p + "post_attention_layernorm.weight"])
            gate = h @ W[p + "mlp.gate_proj.weight"].T
            up = h @ W[p + "mlp.up_proj.weight"].T
            x = x + (gate / (1 + np.exp(-gate)) * up) @ W[p + "mlp.down_proj.weight"].T
        x = self.rms(x, W["model.norm.weight"])
        return x @ W["model.embed_tokens.weight"].T, new_cache     # 词嵌入绑定

    def generate(self, ids, n, T_temp=0.0, rng=None):
        logits, cache = self.forward(ids)
        out = []
        for _ in range(n):
            if T_temp <= 1e-8:
                nxt = int(np.argmax(logits[-1]))
            else:
                p = softmax(logits[-1] / T_temp)
                nxt = int(rng.choice(len(p), p=p))
            out.append(nxt)
            logits, cache = self.forward([nxt], cache)
        return list(ids) + out

In [ ]:
# ===== 点火(讲义 §4 全部数字) =====
tok = Tokenizer(json.load(open(f"{SNAP}/tokenizer.json", encoding="utf-8")))
for s in ["The Journey to the West", "孙悟空 has 72 transformations!", "Hello,  world."]:
    ids = tok.encode(s)
    print(f"encode({s!r}) = {len(ids)} tok | 往返无损: {tok.decode(ids) == s}")

t0 = time.time()
model = SmolLM2()
n_par = sum(v.size for v in model.W.values())
print(f"\n加载 {len(model.W)} 张量 / {n_par:,} 参数"
      f"(fp32 {sum(v.nbytes for v in model.W.values())/1024**3:.2f} GB),{model.load_s:.1f}s")

prompt = "The Journey to the West tells the story of"
ids = tok.encode(prompt)
logits, _ = model.forward(ids)
t5 = np.argsort(-logits[-1])[:5]
print(f"prefill {len(ids)} tok | 下一 token 前5: {[tok.id_to_tok[i] for i in t5]}")

t0 = time.time()
gen = model.generate(ids, 30, T_temp=0.0)
print(f"\n[贪心 30 tok,{30/(time.time()-t0):.1f} tok/s 含 prefill]:")
print("  " + tok.decode(gen))

logits_a, cache = model.forward(ids)
nxt = int(np.argmax(logits_a[-1]))
l1, _ = model.forward([nxt], cache)
l2, _ = model.forward(ids + [nxt])
print(f"\n[18课验收] KV cache vs 全量重算 max|diff| = {np.abs(l1[-1]-l2[-1]).max():.2e}")

# chat:官方模板逐字(首轮系统提示词 + 每段换行)
im_s, im_e = 1, 2
SYS = "You are a helpful AI assistant named SmolLM, trained by Hugging Face"
q = "Tell me about the Journey to the West in two sentences."
chat = ([im_s] + tok.encode("system\n" + SYS) + [im_e]
        + [im_s] + tok.encode("user\n" + q) + [im_e]
        + [im_s] + tok.encode("assistant\n"))
t0 = time.time()
ans = model.generate(chat, 60, T_temp=0.6, rng=np.random.default_rng(3))
print(f"\n[chat 问答,T=0.6,60 tok,{time.time()-t0:.1f}s]:")
print("  " + tok.decode(ans[len(chat):]).split("<|im_end|>")[0])

t0 = time.time()
gen2 = model.generate(ids, 30, T_temp=0.8, rng=np.random.default_rng(7))
print(f"\n[T=0.8 采样 30 tok,{time.time()-t0:.1f}s]:")
print("  " + tok.decode(gen2))

In [ ]:
# ---- 作业现场 ----

# TODO(C2): 用任何独立实现(如临时装 MLX)对账 top-3

# TODO(C3): 接 31 课 int4:量化→反量化→生成,对比质量与速度

# TODO(D1): fp32 0.50GB + KV 0.04GB 的并发账

# TODO(E1): theta ∈ {1e4,1e5,1e6} 的长上下文行为
